# LogicMateV1 - Prototipo de Proyecto de Grado

In [2]:
import os
import cv2
import uuid
import json
import shutil
import logging
import numpy as np
import pandas as pd
from pathlib import Path
# PySceneDetect
from enum import Enum
from scenedetect.platform import init_logger
from scenedetect.backends import AVAILABLE_BACKENDS
from scenedetect.scene_manager import get_scenes_from_cuts, Interpolation, save_images
from scenedetect import StatsManager,ContentDetector, AdaptiveDetector, SceneManager,detect, split_video_ffmpeg, open_video
# Yolocv12
from IPython.display import Image
import matplotlib.pyplot as plt
import supervision as sv
from ultralytics import YOLO
# DINO
import torch
from PIL import Image
from sklearn.manifold import TSNE
from sklearn.metrics.pairwise import cosine_similarity
from transformers import AutoImageProcessor, AutoModel
# PaddleOCR
from paddleocr import PaddleOCR, draw_ocr


/home/zeus/miniconda3/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Configuración básica
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# Funciones auxiliares
def directory_exists(directory: str) -> bool:
    path = Path(directory)
    return path.exists() and path.is_dir()

def create_directory(directory: str) -> bool:
    path = Path(directory)
    try:
        path.mkdir(parents=True)
        return True
    except Exception as e:
        logging.error(e)
        return False

def sort_predictions_by_x(predictions):
    return sorted(predictions, key=lambda obj: obj["x"])

def sort_predictions_by_y(predictions):
    return sorted(predictions, key=lambda obj: obj["y"])

# PySceneDetect
def print_scenes(scene_list: list):
    for i, (start_time, end_time) in enumerate(scene_list, start=1):
        print("Escena {}: {} - {}".format(i, start_time.get_timecode(), end_time.get_timecode()))

def on_new_scene(frame_img: np.ndarray, frame_num: int):
    logging.info("Escena detectada: {} - {}".format(frame_num, frame_num + 1))

# YOLOv12
def convert_detections_to_json(detections):
    formatted_predictions = []

    for i, (xyxy, conf, cls) in enumerate(zip(detections.xyxy, detections.confidence, detections.class_id)):
        x_min, y_min, x_max, y_max = xyxy
        
        width = x_max - x_min
        height = y_max - y_min
        x_center = x_min + width / 2
        y_center = y_min + height / 2

        detection_id = str(uuid.uuid4())

        formatted_predictions.append({
            "x": round(float(x_center), 2),
            "y": round(float(y_center), 2),
            "width": round(float(width), 2),
            "height": round(float(height), 2),
            "confidence": round(float(conf), 3),
            "class": "code_snippet",
            "class_id": int(cls),
            "detection_id": detection_id
        })

    return {"predictions": formatted_predictions}

def analyze_images_with_yolo(path: Path, yolo_model):
    valid_images_paths = []
    accepted_image_extensions = {".png"}
    
    image_files = [f for f in path.iterdir() if f.suffix.lower() in accepted_image_extensions]
    logging.info(f"Analizando {len(image_files)} archivos de imagen...")

    for image_path in image_files:
        logging.info(f"Analizando {image_path.name}...")
        image = cv2.imread(str(image_path))
        if image is None:
            logging.error(f"No se pudo cargar la imagen '{image_path.name}'.")
            continue

        detection_results = yolo_model(image, verbose=False)[0] 
        detections = sv.Detections.from_ultralytics(detection_results).with_nms()
        annotated_image = BOX_ANNOTATOR.annotate(scene=image, detections=detections)
        annotated_image = LABEL_ANNOTATOR.annotate(scene=image, detections=detections)
        predictions = convert_detections_to_json(detections)
        if len(predictions["predictions"]) > MIN_PREDICTION_THRESHOLD:
            # sv.plot_image(annotated_image)
            valid_images_paths.append(image_path)

        logging.info(f"Analisis de {image_path.name} finalizado.")

    logging.info(f"Analisis de {len(image_files)} archivos de imagen finalizado.")
    logging.info(f"Se encontraron {len(valid_images_paths)} archivos de imagen con códigos de algoritmos.")
    return valid_images_paths

def get_preddictions_with_yolo(valid_images_paths: list, yolo_model):
    predictions_list = []
    for image_path in valid_images_paths:
        image_name = os.path.basename(image_path)
        logging.info(f"Extrayendo predicciones de {image_name}...")
        image = cv2.imread(str(image_path))
        if image is None:
            logging.error(f"No se pudo cargar la imagen '{image_name}'.")
            continue

        detection_results = yolo_model(image, verbose=False)[0] 
        predictions = convert_detections_to_json(sv.Detections.from_ultralytics(detection_results).with_nms())
        predictions_list.append(predictions)

    return predictions_list

# DINO
def extract_features(image, model, processor):
    image = Image.open(image).convert("RGB")
    inputs = processor(images=image, return_tensors="pt")
    
    with torch.no_grad():
        outputs = model(**inputs)
        features = outputs.last_hidden_state.mean(dim=1).detach().cpu().numpy()
    
    return features


def extract_features_from_images(dataset_path: str, image_files: list, model, processor) -> list:
    features_list = []
    for image_file in image_files:
        image_path = os.path.join(dataset_path, image_file)
        logging.info(f"Extrayendo features de {image_path}...")
        try:
            features = extract_features(image_path, model, processor)
            features_list.append(features)
        except FileNotFoundError as e:
            logging.error(e)
    return features_list


def filter_similar_images(flattened_features, similarity_matrix, threshold=0.98):
    n = len(flattened_features)
    selected_indices = []
    
    for i in range(n):
        keep = True
        for j in selected_indices:
            if similarity_matrix[i, j] > threshold:
                keep = False
                break
        if keep:
            selected_indices.append(i)
    
    return selected_indices

# PaddleOCR
def extract_text_results(img_path):
    ocr = PaddleOCR(
        use_angle_cls=True,
        lang='en',
        det_db_box_thresh=0.1,
        det_db_unclip_ratio=3.0,
        rec_algorithm='CRNN'
    )
    result = ocr.ocr(img_path, cls=True)
    
    return result

def extract_text_from_resutls(results):
    if not results or results[0] is None:
        logging.error("No se encontraron resultados para extraer texto.")
        return ""
    try:
        extracted_text = "\n".join([line[1][0] for line in results[0]])
    except Exception as e:
        logging.error("Error al extraer texto: %s", e)
        return ""
    return extracted_text

def get_cropped_image(image, predictions):
    x_center = predictions["x"]
    y_center = predictions["y"]
    w = predictions["width"]
    h = predictions["height"]

    x1 = int(x_center - w/2)
    y1 = int(y_center - h/2)
    x2 = int(x_center + w/2)
    y2 = int(y_center + h/2)

    x1 = max(0, x1)
    y1 = max(0, y1)
    x2 = min(image.shape[1], x2)
    y2 = min(image.shape[0], y2)

    cropped_image = image[y1:y2, x1:x2]

    return cropped_image

def scale_image(image, scale):
    return cv2.resize(image, None, fx=3.0, fy=3.0, interpolation=cv2.INTER_LANCZOS4)

def show_highlighted_text_from_image(image, result):
    if image is None:
        logging.error(f"No se pudo cargar la imagen '{img_path}'.")
        return

    if not result or not result[0]:
        logging.error("No se encontraron resultados.")
        return

    result = result[0]

    boxes = [line[0] for line in result]
    txts = [line[1][0] for line in result]
    scores = [float(line[1][1]) for line in result]

    iamge = draw_ocr(image, boxes, txts, scores, font_path="/teamspace/studios/this_studio/simfang.ttf")
    img_rgb = cv2.cvtColor(iamge, cv2.COLOR_BGR2RGB)
    plt.imshow(iamge)
    plt.axis("off")
    plt.show()

# Constantes
# Video
VIDEO_PATH = '/teamspace/studios/this_studio/test_01.mp4'
OUTPUT_DIR = '/teamspace/studios/this_studio/_Results'
VALID_IMAGES_DIR = '/teamspace/studios/this_studio/valid_results'
VALID_CROPPED_IMAGES_DIR = '/teamspace/studios/this_studio/valid_cropped_results'

# PySceneDetect
ADAPTATIVE_THRESHOLD = 15.0
MIN_SCENE_LEN = 20
WINDOW_WIDTH = 8
MIN_CONTENT_VAL = 12
LUMA_ONLY = True
KERNEL_SIZE = None

# Yolov12
MODEL_DIR = '/teamspace/studios/this_studio/LogicMate-v1/notebooks/__yolov12/yolov12/runs/detect/train7/weights/best.pt'
BOX_ANNOTATOR = sv.BoxAnnotator()
LABEL_ANNOTATOR = sv.LabelAnnotator()
MIN_PREDICTION_THRESHOLD = 4 # Evaluar más a profundidad si es suficiente/necesario o mejor ResNet34
CODE_CLASSES=["code_snippet"]

# DINO
DINO_PRETRAINED_MODEL = 'facebook/dinov2-base'
MIN_SIMILARITY_THRESHOLD = 0.99

# Inialización de PySceneDetect
logging.info("Inicializaddo detección de escenas")
print("Inicializando...")
try:
    init_logger(log_level=logging.INFO)
    video = open_video(VIDEO_PATH)
    scene_manager = SceneManager(stats_manager=StatsManager())
    scene_manager.add_detector(AdaptiveDetector(adaptive_threshold=ADAPTATIVE_THRESHOLD, min_scene_len=MIN_SCENE_LEN, window_width=WINDOW_WIDTH, min_content_val=MIN_CONTENT_VAL, luma_only=LUMA_ONLY, kernel_size=KERNEL_SIZE))
    Interpolation(4)
    logging.info("Comenzando detección de escenas...")
    scene_manager.detect_scenes(video, callback=on_new_scene)
    scene_list = scene_manager.get_scene_list()
except Exception as e:
    logging.error(e)
    exit(1)

logging.info("Finalizando detección de escenas")
logging.info("Escenas detectadas: %d", len(scene_list))

logging.info("Guardando escenas temporales...")
try:
    if(directory_exists(OUTPUT_DIR)):
        save_images(video=video, scene_list=scene_list, output_dir=OUTPUT_DIR, image_extension='png', encoder_param=100)
    else:
        create_directory(OUTPUT_DIR)
        save_images(video=video, scene_list=scene_list, output_dir=OUTPUT_DIR, image_extension='png', encoder_param=100)
except Exception as e:
    logging.error(e)

logging.info("Validando escenas con códigos de algoritmos con YOLOv12...")

# Usando Yolov12 para validar si contiene códigos de algoritmos
try:
    logging.info("Cargando modelo de YOLOv12...")
    code_detector_model = YOLO(MODEL_DIR)
    logging.info("Modelo de YOLOv12 cargado correctamente")
    logging.info("Comenzando validación de escenas...")
    valid_images_paths = analyze_images_with_yolo(path=Path(OUTPUT_DIR), yolo_model=code_detector_model)
    logging.info("Finalizando validación de escenas...")
    logging.info("Guardando imagenes validadas...")
    if(directory_exists(VALID_IMAGES_DIR)):
        for valid_image_path in valid_images_paths:
            shutil.copy(valid_image_path, VALID_IMAGES_DIR)
    else:
        create_directory(VALID_IMAGES_DIR)
        for valid_image_path in valid_images_paths:
            shutil.copy(valid_image_path, VALID_IMAGES_DIR)
       
    logging.info("Imagenes validadas guardadas correctamente")
except Exception as e:
    logging.error(e)

logging.info("Finalizando validación de escenas con códigos de algoritmos con YOLOv12")

logging.info("Inicializando similitud entre escenas con DINO...")
try:
    logging.info("Cargando modelo de DINO...")
    processor = AutoImageProcessor.from_pretrained(DINO_PRETRAINED_MODEL)
    model = AutoModel.from_pretrained(DINO_PRETRAINED_MODEL)
    all_images = os.listdir(VALID_IMAGES_DIR)
    logging.info("Cargando imágenes...")
    image_files = [f for f in all_images if f.endswith(".png")]
    logging.info("Extrayendo características de imágenes...")
    features_list = extract_features_from_images(dataset_path=VALID_IMAGES_DIR, image_files=image_files, model=model, processor=processor)
    flattened_features = [feature.flatten() for feature in features_list]
    logging.info("Cargando similitudes entre escenas...")
    similarity_matrix = cosine_similarity(flattened_features)
    selected_indices = filter_similar_images(flattened_features, similarity_matrix, threshold=MIN_SIMILARITY_THRESHOLD)
    logging.info("Índices de imágenes seleccionadas: %s", selected_indices)
    filtered_image_files = [image_files[i] for i in selected_indices]
    logging.info("Guardando imagenes filtradas...")
except Exception as e:
    logging.error(e)
logging.info("Finalizando similitud entre escenas con DINO")

logging.info("Inicializando detección de código de algoritmos de las imagenes filtradas...")
try:
    full_image_paths = [os.path.join(OUTPUT_DIR, image_file) for image_file in filtered_image_files]
    all_images_predictions = get_preddictions_with_yolo(valid_images_paths=full_image_paths, yolo_model=code_detector_model)
except Exception as e:
    logging.error(e)
logging.info("Finalizando detección de código de algoritmos de las imagenes filtradas")

logging.info("Inicializando extracción de código de algoritmos...")
try:
    for index, image_path in enumerate(full_image_paths):
        extracted_code = []
        logging.info(f"Cargando imagen {index+1} de {len(full_image_paths)}")
        image = cv2.imread(image_path)
        predictions = all_images_predictions[index]
        print(predictions["predictions"])
        logging.info(f"Ordenando predicciones de imagen {index+1} de {len(full_image_paths)}")
        sorted_predictions = sort_predictions_by_y(predictions["predictions"])
        print(sorted_predictions)

        for index2, prediction in enumerate(sorted_predictions):
            cropped_image = get_cropped_image(image, prediction)
            logging.info(f"Escalando imagen {index+1} de {len(full_image_paths)}, predicción {index2+1}")
            scaled_image = scale_image(cropped_image, 3.0)
            logging.info(f"Guardando cortada imagen {index+1} de {len(full_image_paths)}, predicción {index2+1}")
            cropped_image_name = f"cropped_image_{index+1}_{index2+1}.png"
            path_to_cropped_image = os.path.join(VALID_CROPPED_IMAGES_DIR, cropped_image_name)
            cv2.imwrite(path_to_cropped_image, scaled_image)
            logging.info(f"Extrayendo texto de imagen {index+1} de {len(full_image_paths)}, predicción {index2+1}")
            extracted_text_results = extract_text_results(path_to_cropped_image)
            logging.info(f"Guardando resultado de extracción de texto de imagen {index+1} de {len(full_image_paths)}, predicción {index2+1}")
            extracted_text = extract_text_from_resutls(extracted_text_results)
            extracted_code.append(extracted_text)
            logging.info(f"Mostrando resultado de extracción de texto de imagen {index+1} de {len(full_image_paths)}, predicción {index2+1}")
            show_highlighted_text_from_image(scaled_image, extracted_text_results)
            logging.info(f"Texto extraido: {extracted_text} de {image_path}, predicción {index2+1}")

        logging.info(f"Codigo extraido: {extracted_code} de {image_path}")
except Exception as e:
    logging.error(e)
logging.info("Finalizando extracción de código de algoritmos")







[2025-03-17 04:23:18,642] [    INFO] scene_manager.py:1335 - Detecting scenes...


Inicializando...


[2025-03-17 04:24:32,382] [    INFO] scene_manager.py:515 - Saving 3 images per scene [format=png] /teamspace/studios/this_studio/_Results 


KeyboardInterrupt: 